# QR Code Detection — Augmented Pipeline (Offline + Online)
**Offline Augmentation (Albumentations) → Train YOLOv8 30 epoch với Online Aug → Detect → Decode → CSV**

> Bật GPU trước khi chạy: Settings → Accelerator → GPU P100

## Chiến lược Augmentation
| | Offline Aug | Online Aug |
|---|---|---|
| **Khi nào** | Trước khi train (1 lần) | Mỗi batch lúc train (ngẫu nhiên) |
| **Cách** | Albumentations tạo ảnh mới | YOLO built-in mosaic, flip, hsv |
| **Tác dụng** | Tăng số lượng ảnh thực sự | Đa dạng hóa mỗi epoch |
| **Kết hợp** | ✅ 2 cơ chế độc lập, cộng hưởng nhau |

## Cell 1a — Cài thư viện (chạy 1 lần, sau đó Restart Session)

In [ ]:
# ── Cell 1a: Cài thư viện ────────────────────────────────────────────────────
# Chạy cell này → đợi xong → bấm Restart Session → chạy Cell 1b trở đi
import subprocess, sys

print('📦 Cài libzbar0...')
subprocess.run(['apt-get', 'install', '-y', 'libzbar0'], capture_output=True)

print('📦 Cài PyTorch cu118 (tương thích P100)...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    '--force-reinstall', '--no-deps',
    '--target', '/usr/local/lib/python3.12/dist-packages',
    'torch', 'torchvision', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/cu118'
], check=True)

print('📦 Cài các thư viện còn lại...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'ultralytics', 'roboflow', 'pyzbar', 'albumentations',
    'zxing-cpp', 'qrcode[pil]'
], check=True)

print()
print('=' * 50)
print('✅ Cài xong!')
print('👉 Bây giờ bấm: Run > Restart Session')
print('   Sau đó chạy từ Cell 1b trở xuống')
print('=' * 50)

## Cell 1b — Import (chạy sau khi Restart Session)

In [ ]:
# ── Cell 1b: Import sau khi restart ─────────────────────────────────────────
import os, cv2, json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import yaml
import torch
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO
from pyzbar import pyzbar
from PIL import Image, ImageFilter, ImageEnhance
import albumentations as A
from tqdm import tqdm

try:
    import zxingcpp
    ZXING_OK = True
    print('✅ zxing-cpp available')
except ImportError:
    ZXING_OK = False
    print('⚠️  zxing-cpp not available, fallback to pyzbar only')

warnings.filterwarnings('ignore')

# Verify GPU
print(f'\n✅ PyTorch {torch.__version__}')
print(f'   CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU            : {torch.cuda.get_device_name(0)}')
    print(f'   CUDA version   : {torch.version.cuda}')
    cc = torch.cuda.get_device_capability(0)
    print(f'   Compute Cap    : {cc[0]}.{cc[1]}')
    # Quick kernel test
    try:
        _ = torch.ones(2, 2).cuda() @ torch.ones(2, 2).cuda()
        print('   Kernel test    : ✅ PASSED')
    except Exception as e:
        print(f'   Kernel test    : ❌ FAILED — {e}')
        print('   → PyTorch chưa được override, thử chạy lại Cell 1a')
print('\n✅ Import xong! Sẵn sàng chạy các cell tiếp theo.')

## Cell 2 — Tải dataset từ Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="Tjh9KiTrl48QgJ9NbNLk")
project = rf.workspace("lihang-xu").project("qr-code-oerhe")
dataset = project.version(1).download("yolov8", location="/kaggle/working/dataset")

DATASET_PATH = dataset.location
print(f'✅ Dataset tại: {DATASET_PATH}')

train_imgs = list(Path(DATASET_PATH + '/train/images').glob('*.jpg')) + \
             list(Path(DATASET_PATH + '/train/images').glob('*.png'))
print(f'📊 Ảnh train gốc: {len(train_imgs)}')

## Cell 2b — Tạo Synthetic Multi-QR Dataset

**Mục tiêu:** Sinh 550 ảnh tổng hợp, mỗi ảnh chứa 6–12 QR code ở các vị trí ngẫu nhiên.

**Chiến lược:**
- Dùng `qrcode` để render QR từ các chuỗi text đa dạng (URL, SKU, UUID, text)
- Dán QR lên background ngẫu nhiên (màu solid, noise, gradient, kệ hàng)
- Random: kích thước QR, góc xoay, độ mờ, vị trí
- Tự động sinh label YOLO tương ứng
- Merge với dataset Roboflow gốc → augment chung

In [ ]:
import qrcode
import random, uuid

# ── Cấu hình ────────────────────────────────────────────────────────────────
SYNTH_DIR     = Path('/kaggle/working/synthetic_qr')
SYNTH_IMG_DIR = SYNTH_DIR / 'images'
SYNTH_LBL_DIR = SYNTH_DIR / 'labels'
SYNTH_IMG_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_LBL_DIR.mkdir(parents=True, exist_ok=True)

NUM_IMAGES       = 550
QR_PER_IMAGE_MIN = 6
QR_PER_IMAGE_MAX = 12
CANVAS_SIZE      = 1024
QR_SIZE_MIN      = 80
QR_SIZE_MAX      = 180
MAX_OVERLAP      = 0.15

def random_qr_content():
    r = random.random()
    if r < 0.25:
        domains = ['example.com', 'shop.vn', 'warehouse.net', 'scan.io', 'qr-track.com']
        return f"https://{random.choice(domains)}/item/{uuid.uuid4().hex[:8]}"
    elif r < 0.50:
        prefix = random.choice(['SKU', 'WH', 'PROD', 'ITEM', 'LOT', 'SN'])
        return f"{prefix}-{random.randint(10000,99999)}-{uuid.uuid4().hex[:6].upper()}"
    elif r < 0.70:
        return str(uuid.uuid4())
    elif r < 0.85:
        serials = ['SN', 'MSP', 'MH', 'ID']
        return f"{random.choice(serials)}:{random.randint(100000,999999)}"
    else:
        return ''.join([str(random.randint(0, 9)) for _ in range(13)])

def render_qr(content, size_px):
    qr = qrcode.QRCode(version=None,
                       error_correction=qrcode.constants.ERROR_CORRECT_M,
                       box_size=10, border=2)
    qr.add_data(content)
    qr.make(fit=True)
    fg = random.choice([(0,0,0), (20,20,80), (60,20,20), (0,50,0)])
    bg = random.choice([(255,255,255), (240,240,240), (255,255,240), (245,245,255)])
    img = qr.make_image(fill_color=fg, back_color=bg).convert('RGB')
    return img.resize((size_px, size_px), Image.NEAREST)

def make_background(size):
    bg_type = random.choice(['solid', 'noise', 'gradient', 'shelf'])
    if bg_type == 'solid':
        arr = np.full((size, size, 3),
                      [random.randint(180,255), random.randint(180,255), random.randint(180,255)],
                      dtype=np.uint8)
    elif bg_type == 'noise':
        arr = np.random.randint(160, 230, (size, size, 3), dtype=np.uint8)
    elif bg_type == 'gradient':
        arr = np.zeros((size, size, 3), dtype=np.uint8)
        c1 = np.array([random.randint(180,255)]*3)
        c2 = np.array([random.randint(180,255)]*3)
        for row in range(size):
            arr[row] = (c1*(1-row/size) + c2*(row/size)).astype(np.uint8)
    else:
        arr = np.full((size, size, 3), 220, dtype=np.uint8)
        for y in range(0, size, random.randint(80, 160)):
            arr[y:y+4, :] = [150, 130, 110]
        for x in range(0, size, random.randint(100, 200)):
            arr[:, x:x+3] = [170, 150, 130]
    return Image.fromarray(arr)

def iou_area(b1, b2):
    ix1, iy1 = max(b1[0],b2[0]), max(b1[1],b2[1])
    ix2, iy2 = min(b1[2],b2[2]), min(b1[3],b2[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0: return 0.0
    return inter / ((b1[2]-b1[0])*(b1[3]-b1[1]))

def generate_synthetic_image(img_idx):
    canvas = make_background(CANVAS_SIZE)
    n_qr   = random.randint(QR_PER_IMAGE_MIN, QR_PER_IMAGE_MAX)
    placed, labels = [], []

    for _ in range(n_qr * 5):
        if len(placed) >= n_qr: break
        sz = random.randint(QR_SIZE_MIN, QR_SIZE_MAX)
        margin = 10
        x1 = random.randint(margin, CANVAS_SIZE - sz - margin)
        y1 = random.randint(margin, CANVAS_SIZE - sz - margin)
        x2, y2 = x1+sz, y1+sz

        if any(iou_area((x1,y1,x2,y2), pb) > MAX_OVERLAP for pb in placed):
            continue

        try:
            qr_img = render_qr(random_qr_content(), sz)
        except Exception:
            continue

        angle = random.choice([0,0,0,5,-5,10,-10,15,-15,20,-20])
        if angle != 0:
            qr_img = qr_img.rotate(angle, expand=True, fillcolor=(255,255,255))
            nw, nh = qr_img.size
            qr_img = qr_img.crop(((nw-sz)//2, (nh-sz)//2, (nw-sz)//2+sz, (nh-sz)//2+sz))

        if random.random() < 0.25:
            qr_img = qr_img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
        if random.random() < 0.20:
            qr_img = ImageEnhance.Brightness(qr_img).enhance(random.uniform(0.6, 1.3))

        canvas.paste(qr_img, (x1, y1))
        placed.append((x1, y1, x2, y2))
        labels.append(f"0 {(x1+x2)/2/CANVAS_SIZE:.6f} {(y1+y2)/2/CANVAS_SIZE:.6f} "
                      f"{sz/CANVAS_SIZE:.6f} {sz/CANVAS_SIZE:.6f}")

    if len(placed) < 2: return False

    if random.random() < 0.3:
        arr  = np.array(canvas).astype(np.int16)
        arr  = np.clip(arr + np.random.randint(-15,15,arr.shape,dtype=np.int16), 0, 255).astype(np.uint8)
        canvas = Image.fromarray(arr)

    name = f"synth_{img_idx:05d}"
    canvas.save(str(SYNTH_IMG_DIR / f"{name}.jpg"), quality=92)
    with open(SYNTH_LBL_DIR / f"{name}.txt", 'w') as f:
        f.write('\n'.join(labels))
    return True

# ── Chạy sinh dataset ────────────────────────────────────────────────────────
print(f'🏭 Bắt đầu sinh {NUM_IMAGES} ảnh synthetic multi-QR...')
print(f'   Mỗi ảnh: {QR_PER_IMAGE_MIN}–{QR_PER_IMAGE_MAX} QR | Canvas: {CANVAS_SIZE}×{CANVAS_SIZE}px')

success = 0
for i in tqdm(range(NUM_IMAGES + 50), desc='Generating'):
    if success >= NUM_IMAGES: break
    if generate_synthetic_image(success): success += 1

total_qr = sum(len(open(l).readlines()) for l in SYNTH_LBL_DIR.glob('*.txt'))
print(f'\n✅ Synthetic dataset sẵn sàng!')
print(f'   Ảnh: {success} | Tổng QR bbox: {total_qr} | TB: {total_qr/max(1,success):.1f}/ảnh')

## Cell 2c — Merge Synthetic vào Dataset Roboflow

In [ ]:
TRAIN_IMG_DIR   = Path(DATASET_PATH) / 'train' / 'images'
TRAIN_LABEL_DIR = Path(DATASET_PATH) / 'train' / 'labels'
OUTPUT_DIR      = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

before = len(list(TRAIN_IMG_DIR.glob('*.*')))
synth_imgs = list(SYNTH_IMG_DIR.glob('*.jpg'))
print(f'📥 Copy {len(synth_imgs)} ảnh synthetic vào train set...')

for img_path in tqdm(synth_imgs, desc='Merging'):
    shutil.copy2(img_path, TRAIN_IMG_DIR / img_path.name)
    lbl = SYNTH_LBL_DIR / (img_path.stem + '.txt')
    if lbl.exists():
        shutil.copy2(lbl, TRAIN_LABEL_DIR / lbl.name)

after = len(list(TRAIN_IMG_DIR.glob('*.*')))
print(f'\n✅ Merge xong!')
print(f'   Roboflow gốc: {before} | Synthetic: +{len(synth_imgs)} | Tổng: {after}')

# Preview 3 ảnh synthetic
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Preview — Synthetic Multi-QR Images', fontsize=13)
for i, sp in enumerate(sorted(SYNTH_IMG_DIR.glob('*.jpg'))[:3]):
    img = cv2.imread(str(sp))
    lbl = SYNTH_LBL_DIR / (sp.stem + '.txt')
    H, W = img.shape[:2]
    if lbl.exists():
        for line in open(lbl):
            p = line.strip().split()
            if len(p) < 5: continue
            cx,cy,bw,bh = map(float,p[1:5])
            cv2.rectangle(img,
                          (int((cx-bw/2)*W), int((cy-bh/2)*H)),
                          (int((cx+bw/2)*W), int((cy+bh/2)*H)),
                          (0,200,80), 3)
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    n = len(open(lbl).readlines()) if lbl.exists() else 0
    axes[i].set_title(f'{sp.name} | {n} QR', fontsize=9)
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/00_synthetic_preview.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Preview xong — sẵn sàng Offline Augmentation')

## Cell 3 — Offline Augmentation (Albumentations)

**Pipeline aug offline gồm:**
- Blur, Gaussian noise → giả lập camera kém
- Brightness/Contrast → giả lập ánh sáng thay đổi
- Perspective transform → giả lập góc chụp
- CLAHE → cải thiện contrast cục bộ
- Rotate nhẹ (±15°) → giả lập QR code nghiêng

In [ ]:
def load_yolo_labels(label_path, img_w, img_h):
    bboxes, class_labels = [], []
    if not os.path.exists(label_path): return bboxes, class_labels
    for line in open(label_path):
        p = line.strip().split()
        if len(p) < 5: continue
        cls = int(float(p[0]))
        cx,cy,w,h = map(float, p[1:5])
        bboxes.append([(cx-w/2)*img_w, (cy-h/2)*img_h, (cx+w/2)*img_w, (cy+h/2)*img_h])
        class_labels.append(cls)
    return bboxes, class_labels

def save_yolo_labels(save_path, bboxes, class_labels, img_w, img_h):
    with open(save_path, 'w') as f:
        for bbox, cls in zip(bboxes, class_labels):
            x1,y1,x2,y2 = bbox
            cx = np.clip((x1+x2)/2/img_w, 0, 1)
            cy = np.clip((y1+y2)/2/img_h, 0, 1)
            w  = np.clip((x2-x1)/img_w,   0, 1)
            h  = np.clip((y2-y1)/img_h,   0, 1)
            if w > 0.01 and h > 0.01:
                f.write(f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')

offline_aug = A.Compose([
    A.OneOf([
        A.GaussianBlur(blur_limit=(3,7), p=1.0),
        A.MotionBlur(blur_limit=7,       p=1.0),
        A.MedianBlur(blur_limit=5,       p=1.0),
    ], p=0.5),
    A.GaussNoise(var_limit=(10,50), p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.4),
    A.Perspective(scale=(0.05,0.1), p=0.3),
    A.RandomGamma(gamma_limit=(80,120), p=0.3),
], bbox_params=A.BboxParams(
    format='pascal_voc', label_fields=['class_labels'],
    min_area=100, min_visibility=0.3
))

img_files    = sorted(list(TRAIN_IMG_DIR.glob('*.jpg')) + list(TRAIN_IMG_DIR.glob('*.png')))
AUG_MULTIPLIER = 2
print(f'🔄 Offline augmentation...')
print(f'   Ảnh gốc: {len(img_files)} | Tạo thêm: {len(img_files)*AUG_MULTIPLIER} | Tổng: {len(img_files)*(AUG_MULTIPLIER+1)}')

augmented_count = error_count = 0
for img_path in tqdm(img_files, desc='Augmenting'):
    img = cv2.imread(str(img_path))
    if img is None: continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    bboxes, class_labels = load_yolo_labels(str(TRAIN_LABEL_DIR/(img_path.stem+'.txt')), w, h)

    for aug_idx in range(AUG_MULTIPLIER):
        try:
            r = offline_aug(image=img_rgb,
                            bboxes=bboxes if bboxes else [],
                            class_labels=class_labels if class_labels else [])
            name = f"{img_path.stem}_aug{aug_idx}"
            cv2.imwrite(str(TRAIN_IMG_DIR/f"{name}.jpg"), cv2.cvtColor(r['image'], cv2.COLOR_RGB2BGR))
            save_yolo_labels(str(TRAIN_LABEL_DIR/f"{name}.txt"), r['bboxes'], r['class_labels'], w, h)
            augmented_count += 1
        except Exception:
            error_count += 1

all_imgs = list(TRAIN_IMG_DIR.glob('*.jpg')) + list(TRAIN_IMG_DIR.glob('*.png'))
print(f'\n✅ Augmentation xong! Tạo thêm: {augmented_count} | Lỗi: {error_count} | Tổng: {len(all_imgs)}')

## Cell 4 — Xem trước ảnh augmented

In [ ]:
def draw_yolo_boxes(img_bgr, label_path):
    h, w = img_bgr.shape[:2]
    img_draw = img_bgr.copy()
    if not os.path.exists(label_path): return img_draw
    for line in open(label_path):
        p = line.strip().split()
        if len(p) < 5: continue
        cx,cy,bw,bh = map(float,p[1:5])
        cv2.rectangle(img_draw,
                      (int((cx-bw/2)*w), int((cy-bh/2)*h)),
                      (int((cx+bw/2)*w), int((cy+bh/2)*h)),
                      (0,255,0), 2)
    return img_draw

fig, axes = plt.subplots(2, 4, figsize=(20,10))
fig.suptitle('Offline Augmentation — Gốc (trên) vs Augmented (dưới)', fontsize=14)
orig_imgs = [f for f in sorted(TRAIN_IMG_DIR.glob('*.jpg')) if '_aug' not in f.stem][:4]

for i, orig in enumerate(orig_imgs):
    aug = TRAIN_IMG_DIR / f"{orig.stem}_aug0.jpg"
    axes[0][i].imshow(cv2.cvtColor(draw_yolo_boxes(cv2.imread(str(orig)),
                                                    str(TRAIN_LABEL_DIR/f"{orig.stem}.txt")),
                                   cv2.COLOR_BGR2RGB))
    axes[0][i].set_title(f'Gốc: {orig.name}', fontsize=9); axes[0][i].axis('off')
    if aug.exists():
        axes[1][i].imshow(cv2.cvtColor(draw_yolo_boxes(cv2.imread(str(aug)),
                                                        str(TRAIN_LABEL_DIR/f"{orig.stem}_aug0.txt")),
                                       cv2.COLOR_BGR2RGB))
        axes[1][i].set_title(f'Aug: {aug.name}', fontsize=9)
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_augmentation_preview.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Preview augmentation xong!')

## Cell 5 — Train YOLOv8 (30 epoch + Online Augmentation)

In [ ]:
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
use_half = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 7
print(f'🔧 Device: {device.upper()} | FP16: {use_half}')

yaml_path = '/kaggle/working/qr_data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump({'path': DATASET_PATH, 'train': 'train/images',
               'val': 'valid/images', 'test': 'test/images',
               'nc': 1, 'names': ['QR Code']}, f, default_flow_style=False)
print('📄 data.yaml tạo xong')

model   = YOLO('yolov8n.pt')
results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    batch=16,
    device=device,
    half=use_half,
    name='qr_aug_30ep',
    project='/kaggle/working/runs',
    exist_ok=True,
    patience=10,
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.1,
    flipud=0.1,
    fliplr=0.5,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    save=True,
    plots=True,
    verbose=True,
)
print(f'\n✅ Training xong! Best mAP50: {results.results_dict.get("metrics/mAP50(B)", 0):.4f}')

## Cell 6 — Load model & xem kết quả training

In [ ]:
BEST_PT    = '/kaggle/working/runs/qr_aug_30ep/weights/best.pt'
best_model = YOLO(BEST_PT)
print(f'✅ Load: {BEST_PT}')

train_csv = '/kaggle/working/runs/qr_aug_30ep/results.csv'
if os.path.exists(train_csv):
    df = pd.read_csv(train_csv)
    df.columns = df.columns.str.strip()
    fig, axes = plt.subplots(1, 3, figsize=(18,5))
    fig.suptitle('Training 30 Epoch — Offline + Online Augmentation', fontsize=13)

    axes[0].plot(df['epoch'], df['train/box_loss'], label='Train', color='#e74c3c')
    axes[0].plot(df['epoch'], df['val/box_loss'],   label='Val',   color='#e74c3c', ls='--')
    axes[0].set_title('Box Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(df['epoch'], df['metrics/mAP50(B)'],    label='mAP50',    color='#2ecc71')
    axes[1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', color='#27ae60', ls='--')
    axes[1].set_title('mAP'); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='#3498db')
    axes[2].plot(df['epoch'], df['metrics/recall(B)'],    label='Recall',    color='#9b59b6')
    axes[2].set_title('Precision & Recall'); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/02_training_curves.png', dpi=100, bbox_inches='tight')
    plt.show()

    best_ep = df['metrics/mAP50(B)'].idxmax()
    print(f'\n📊 Best epoch: {int(df.loc[best_ep,"epoch"])}')
    print(f'   mAP50:    {df.loc[best_ep,"metrics/mAP50(B)"]:.4f}')
    print(f'   mAP50-95: {df.loc[best_ep,"metrics/mAP50-95(B)"]:.4f}')
    print(f'   P/R:      {df.iloc[-1]["metrics/precision(B)"]:.3f} / {df.iloc[-1]["metrics/recall(B)"]:.3f}')

## Cell 7 — Detect + Decode trên test set

In [ ]:
TEST_IMG_DIR = Path(DATASET_PATH) / 'test' / 'images'
test_imgs    = sorted(list(TEST_IMG_DIR.glob('*.jpg')) + list(TEST_IMG_DIR.glob('*.png')))
print(f'📂 Test images: {len(test_imgs)}')

def try_decode(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img
    if ZXING_OK:
        r = zxingcpp.read_barcodes(Image.fromarray(gray))
        if r: return r[0].text
    for src in [img if len(img.shape)==3 else cv2.cvtColor(img,cv2.COLOR_GRAY2BGR), gray]:
        d = pyzbar.decode(src)
        if d: return d[0].data.decode('utf-8', errors='replace')
    return None

def generate_variants(crop):
    resized = cv2.resize(crop, (300,300), interpolation=cv2.INTER_CUBIC)
    gray    = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    clahe   = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    v_clahe = clahe.apply(gray)
    _, v_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    k = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], np.float32)
    v_sharp = np.clip(cv2.filter2D(gray,-1,k), 0, 255).astype(np.uint8)
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    _, v_d_otsu = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    v_adapt = cv2.adaptiveThreshold(gray,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,11,2)
    return [resized, v_clahe, v_otsu, v_sharp, v_d_otsu, v_adapt]

def smart_decode(img, x1,y1,x2,y2, pad=15):
    H,W = img.shape[:2]
    crop = img[max(0,y1-pad):min(H,y2+pad), max(0,x1-pad):min(W,x2+pad)]
    if crop.size==0 or crop.shape[0]<10 or crop.shape[1]<10: return '','invalid'
    r = try_decode(crop)
    if r: return r,'direct'
    for v in generate_variants(crop):
        r = try_decode(v)
        if r: return r,'variant'
    return '','failed'

def compute_iou(b1,b2):
    ix1,iy1 = max(b1[0],b2[0]),max(b1[1],b2[1])
    ix2,iy2 = min(b1[2],b2[2]),min(b1[3],b2[3])
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    if inter==0: return 0.0
    return inter/((b1[2]-b1[0])*(b1[3]-b1[1])+(b2[2]-b2[0])*(b2[3]-b2[1])-inter)

def nms(boxes, thresh=0.4):
    keep, sup = [], set()
    for i in sorted(range(len(boxes)), key=lambda i:-boxes[i][4]):
        if i in sup: continue
        keep.append(i)
        for j in range(len(boxes)):
            if j!=i and j not in sup and compute_iou(boxes[i][:4],boxes[j][:4])>thresh:
                sup.add(j)
    return keep

records, decode_stats, session_seen = [], {}, {}

for img_path in tqdm(test_imgs[:50], desc='Detect & Decode'):
    img = cv2.imread(str(img_path))
    if img is None: continue
    res = best_model(img, conf=0.3, verbose=False)[0]

    if len(res.boxes)==0:
        records.append({'image':img_path.name,'qr_count':0,'status':'NOT_DETECTED',
                        'content':'','confidence':0,'decode_method':'','valid':False,
                        'invalid_reason':'','duplicate':False,'dup_note':''})
        continue

    boxes = [(int(b.xyxy[0][0]),int(b.xyxy[0][1]),int(b.xyxy[0][2]),int(b.xyxy[0][3]),float(b.conf[0]))
             for b in res.boxes]

    for i in nms(boxes):
        x1,y1,x2,y2,conf = boxes[i]
        content, method  = smart_decode(img, x1,y1,x2,y2)
        decode_stats[method] = decode_stats.get(method,0)+1

        is_valid = bool(content and len(content.strip())>=4 and
                        sum(c.isprintable() for c in content)/len(content)>=0.85 and
                        conf>=0.5)
        reason = '' if is_valid else ('empty' if not content else 'low_conf' if conf<0.5 else 'garbage')

        is_dup, dup_note = False, ''
        if is_valid:
            if content in session_seen:
                is_dup=True; dup_note=f"seen {session_seen[content]['count']}x"
                session_seen[content]['count']+=1
            else:
                session_seen[content]={'count':1,'first':img_path.name}

        status = ('DETECTED_NOT_DECODED' if not content else
                  'DECODED_INVALID'      if not is_valid else
                  'DECODED_DUPLICATE'    if is_dup else 'DECODED_OK')

        records.append({'image':img_path.name,'qr_count':len(nms(boxes)),'status':status,
                        'content':content,'confidence':round(conf,3),'decode_method':method,
                        'bbox_size':f'{x2-x1}x{y2-y1}','valid':is_valid,
                        'invalid_reason':reason,'duplicate':is_dup,'dup_note':dup_note})

df = pd.DataFrame(records)
OUTPUT_CSV = f'{OUTPUT_DIR}/qr_scan_results.csv'
df.to_csv(OUTPUT_CSV, index=False)

print(f'\n✅ Kết quả {len(test_imgs[:50])} ảnh:')
print(df['status'].value_counts().to_string())
print(f'\n💾 CSV: {OUTPUT_CSV}')

## Cell 8 — Visualize kết quả detect

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18,12))
fig.suptitle('Kết quả Detect + Smart Decode QR Code', fontsize=14)
axes = axes.flatten()

for i, img_path in enumerate(test_imgs[:6]):
    img = cv2.imread(str(img_path))
    if img is None: continue
    res = best_model(img, conf=0.3, verbose=False)[0]
    draw = img.copy()
    title = [img_path.name]
    for box in res.boxes:
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf[0])
        content, method = smart_decode(img,x1,y1,x2,y2)
        color = (0,200,80) if content else (0,140,255)
        cv2.rectangle(draw,(x1,y1),(x2,y2),color,3)
        cv2.putText(draw,f'{conf:.2f}[{method[:6]}]',(x1,max(y1-8,12)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
        title.append(f'✓ {content[:20]}' if content else '✗ fail')
    axes[i].imshow(cv2.cvtColor(draw, cv2.COLOR_BGR2RGB))
    axes[i].set_title('\n'.join(title), fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_detection_results.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Visualize xong!')

## Cell 9 — Export ONNX & Tóm tắt

In [ ]:
best_model.export(format='onnx', imgsz=640)
print('✅ Xuất ONNX xong!')

all_train = list(TRAIN_IMG_DIR.glob('*.jpg')) + list(TRAIN_IMG_DIR.glob('*.png'))
print('\n' + '='*50)
print('📦 TỔNG KẾT PIPELINE')
print('='*50)
print(f'  Ảnh train (sau aug):  {len(all_train)}')
print(f'  Epoch:                30')
print(f'  Model:                YOLOv8n')
print(f'\n📂 Outputs:')
for fp in sorted(Path(OUTPUT_DIR).glob('*')):
    print(f'  {fp.name}  ({fp.stat().st_size/1024:.0f} KB)')